In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load datasets
df_bp = pd.read_csv('blood_pressure.csv')
df_hr = pd.read_csv('heart_rate.csv')
df_hrv = pd.read_csv('hrv_measurements.csv')
df_sleep = pd.read_csv('sleep.csv')
df_wearables = pd.read_csv('wearables.csv')
df_weather = pd.read_csv('weather.csv')
df_participants = pd.read_csv('participants.csv')
df_surveys = pd.read_csv('surveys.csv')
df_scales = pd.read_csv('scales_description.csv')

# Store all DataFrames together so we can analyze them with loops
datasets = {
    'Blood Pressure': df_bp,
    'Heart Rate': df_hr,
    'HRV Measurements': df_hrv,
    'Sleep': df_sleep,
    'Wearables': df_wearables,
    'Weather': df_weather,
    'Participants': df_participants,
    'Surveys': df_surveys,
    'Scale Descriptions': df_scales
}

pd.set_option('display.max_columns', None)

## 1. Dataset Structure and Data Quality

We first examine the basic structure of each dataset.

For each dataset, we want to know:

- How many rows and columns are present?
- Which variables are numerical versus categorical/text?
- How many values are missing?
- Are there duplicated observations?

This provides an initial overview of the size and quality of the data and helps identify datasets or variables that may require additional cleaning.

In [ ]:
# Overall summary of each dataset
overview = []

for name, df in datasets.items():
    overview.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Numerical Variables': len(df.select_dtypes(include=np.number).columns),
        'Categorical/Text Variables': len(df.select_dtypes(include=['object', 'category']).columns),
        'Missing Values': df.isna().sum().sum(),
        'Missing %': round(df.isna().sum().sum() / df.size * 100, 2),
        'Duplicate Rows': df.duplicated().sum()
    })

display(pd.DataFrame(overview))

# More detailed column-level information
for name, df in datasets.items():
    print(f'\n{"="*70}\n{name}\n{"="*70}')
    
    column_info = pd.DataFrame({
        'Data Type': df.dtypes,
        'Unique Values': df.nunique(),
        'Missing Count': df.isna().sum(),
        'Missing %': (df.isna().mean() * 100).round(2)
    })
    
    display(column_info)
    display(df.head())

## 2. Descriptive Statistics

Next, we examine numerical variables using measures of central tendency and dispersion.

**Central tendency** describes the typical value:
- Mean = arithmetic average
- Median = middle value
- Mode = most frequently occurring value

**Dispersion** describes how spread out the data are:
- Range = maximum − minimum
- Variance = average squared deviation from the mean
- Standard deviation = typical distance from the mean
- IQR = spread of the middle 50% of observations

Skewness is also calculated to identify variables with asymmetric distributions. Large positive skew indicates a long right tail, while negative skew indicates a long left tail.

In [ ]:
for name, df in datasets.items():
    
    numeric = df.select_dtypes(include=np.number)
    
    if numeric.empty:
        continue
    
    summary = pd.DataFrame({
        'Mean': numeric.mean(),
        'Median': numeric.median(),
        'Mode': numeric.mode().iloc[0],
        'Min': numeric.min(),
        'Max': numeric.max(),
        'Range': numeric.max() - numeric.min(),
        'Variance': numeric.var(),
        'Std Dev': numeric.std(),
        'IQR': numeric.quantile(0.75) - numeric.quantile(0.25),
        'Skewness': numeric.skew()
    }).round(2)
    
    print(f'\n{"="*70}\n{name}\n{"="*70}')
    display(summary)

## 3. Distributions and Potential Outliers

Histograms show the shape of numerical distributions and can reveal skewness, multiple peaks, or unusual values. Box plots provide another way to visualize the median, spread, and potential outliers.

We also use the IQR rule to identify potential outliers. Values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are flagged.

Importantly, a statistical outlier is not necessarily an error. Extreme physiological measurements may represent real biological variation, unusual clinical states, or measurement problems and therefore require interpretation using domain knowledge.

In [ ]:
def examine_distributions(df, dataset_name, max_columns=8):
    
    numeric_cols = df.select_dtypes(include=np.number).columns[:max_columns]
    
    # Calculate potential outliers
    outlier_results = []
    
    for col in numeric_cols:
        values = df[col].dropna()
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        
        outliers = values[(values < lower) | (values > upper)]
        
        outlier_results.append({
            'Variable': col,
            'Outliers': len(outliers),
            'Outlier %': round(len(outliers) / len(values) * 100, 2),
            'Observed Min': values.min(),
            'Observed Max': values.max()
        })
    
    print(f'Potential outliers: {dataset_name}')
    display(pd.DataFrame(outlier_results))
    
    # Histograms and box plots
    for col in numeric_cols:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[col].dropna(), kde=True)
        plt.title(f'{dataset_name}: Distribution of {col}')
        plt.show()
        
        plt.figure(figsize=(7, 3))
        sns.boxplot(x=df[col])
        plt.title(f'{dataset_name}: Box Plot of {col}')
        plt.show()


# Run on the main physiological datasets
examine_distributions(df_hrv, 'HRV Measurements')
examine_distributions(df_bp, 'Blood Pressure')
examine_distributions(df_sleep, 'Sleep')

## 4. Relationships Between Numerical Variables

Correlation analysis evaluates whether numerical variables tend to change together.

Pearson correlation ranges from −1 to +1:

- Values near +1 indicate a strong positive relationship.
- Values near −1 indicate a strong negative relationship.
- Values near 0 indicate little linear relationship.

Correlation matrices and scatterplots can reveal relationships among physiological, behavioral, and environmental measurements. Correlation identifies association, but does not establish causation.

In [ ]:
def analyze_correlations(df, dataset_name):
    
    numeric = df.select_dtypes(include=np.number)
    
    if numeric.shape[1] < 2:
        return
    
    corr = numeric.corr()
    
    # Correlation heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title(f'{dataset_name}: Correlation Matrix')
    plt.show()
    
    # Extract strongest correlations
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    
    pairs.columns = ['Variable 1', 'Variable 2', 'Correlation']
    pairs['Absolute Correlation'] = pairs['Correlation'].abs()
    
    strong = pairs[pairs['Absolute Correlation'] >= 0.5]
    strong = strong.sort_values('Absolute Correlation', ascending=False)
    
    print(f'Strongest correlations in {dataset_name}:')
    display(strong)


analyze_correlations(df_hrv, 'HRV Measurements')
analyze_correlations(df_wearables, 'Wearables')
analyze_correlations(df_sleep, 'Sleep')
analyze_correlations(df_weather, 'Weather')

## 5. Categorical Variables, Participants, and Longitudinal Structure

Categorical variables are inspected to determine which groups exist and whether inconsistent labels are present.

Because wearable datasets commonly contain repeated measurements from the same individuals, we also search for participant identifiers and date/time variables. This is important because repeated observations from one person should not automatically be treated as independent observations.

Categorical variables can later be compared with numerical measurements using grouped statistics or box plots.

In [ ]:
for name, df in datasets.items():
    
    print(f'\n{"="*70}\n{name}\n{"="*70}')
    
    # Identify likely categorical columns
    categorical = df.select_dtypes(include=['object', 'category']).columns
    
    print('\nCategorical variables:')
    for col in categorical:
        print(f'\n{col}:')
        print(df[col].value_counts(dropna=False).head(10))
    
    # Search for participant/user ID columns
    possible_ids = [
        col for col in df.columns
        if 'id' in col.lower()
        or 'participant' in col.lower()
        or 'user' in col.lower()
    ]
    
    # Search for date/time columns
    possible_dates = [
        col for col in df.columns
        if 'date' in col.lower()
        or 'time' in col.lower()
        or 'timestamp' in col.lower()
    ]
    
    print('\nPossible ID columns:', possible_ids)
    print('Possible date/time columns:', possible_dates)

## 6. Feature Engineering and Preprocessing Considerations

The final step of EDA is determining how the existing variables might be prepared for future analysis.

Potential feature engineering opportunities include:

- Pulse pressure = systolic − diastolic blood pressure
- Mean arterial pressure
- Daily mean, minimum, or maximum heart rate
- Changes in HRV relative to an individual's baseline
- Sleep duration or sleep categories
- Time of day
- Rolling averages of repeated physiological measurements
- Measurements relative to illness or symptom onset

Variables with large differences in scale may require standardization before some machine-learning algorithms. Strongly right-skewed variables may benefit from transformations such as `log(1 + x)`.

These transformations should not be performed automatically. EDA should first identify which variables require them and whether the transformation makes sense based on the meaning of the measurement.

In [ ]:
# Identify variables that may need transformation or scaling
for name, df in datasets.items():
    
    numeric = df.select_dtypes(include=np.number)
    
    if numeric.empty:
        continue
    
    assessment = pd.DataFrame({
        'Minimum': numeric.min(),
        'Maximum': numeric.max(),
        'Mean': numeric.mean(),
        'Std Dev': numeric.std(),
        'Skewness': numeric.skew()
    }).round(2)
    
    # Flag strongly skewed variables
    assessment['Consider Transformation'] = (
        assessment['Skewness'].abs() > 1
    )
    
    print(f'\n{"="*70}\n{name}\n{"="*70}')
    display(assessment)

# Example future feature engineering:
#
# df_bp['pulse_pressure'] = df_bp['systolic'] - df_bp['diastolic']
#
# df_bp['mean_arterial_pressure'] = (
#     df_bp['diastolic'] +
#     (df_bp['systolic'] - df_bp['diastolic']) / 3
# )
#
# A strongly right-skewed variable could be transformed using:
# df['variable_log'] = np.log1p(df['variable'])

## 7. The Cohort: Participants as the Anchor Table

The sections above treat each file on its own. The next sections look at how the files **fit together** as one clinical dataset collected from a defined group of people.

`participants.csv` defines that group. It should hold exactly one row per person, and every other table refers back to it through `user_code`. Before looking at any measurements, we check:

- Is `user_code` a unique key?
- What are the cohort's demographics (gender, age range, country)?
- Are the baseline measurements plausible? Body mass index (BMI) is derived from height (cm) and weight (kg).
- How many participants reported a symptom onset date, and are those dates plausible for a 2020 study?

In [ ]:
# The participants table defines the cohort: one row per person
print('Rows:', len(df_participants), '| Unique user_code values:', df_participants['user_code'].nunique())
print('user_code is a unique key:', df_participants['user_code'].is_unique)

cohort = df_participants.copy()
cohort['bmi'] = cohort['weight'] / (cohort['height'] / 100) ** 2
cohort['symptoms_onset'] = pd.to_datetime(cohort['symptoms_onset'], format='%m/%d/%Y', errors='coerce')

print('\nBaseline body measurements:')
display(cohort[['height', 'weight', 'bmi']].describe().round(1))

has_onset = cohort['symptoms_onset'].notna()
implausible_onset = has_onset & ~cohort['symptoms_onset'].between('2019-12-01', '2020-12-31')
print(f'Participants with a reported symptom onset: {has_onset.sum()} of {len(cohort)}')
print(f'Onset dates outside Dec 2019 - Dec 2020 (likely entry errors): {implausible_onset.sum()}')
display(cohort.loc[implausible_onset, ['user_code', 'symptoms_onset']])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
age_order = sorted(cohort['age_range'].dropna().unique())
sns.countplot(data=cohort, x='age_range', hue='gender', order=age_order, ax=axes[0])
axes[0].set_title('Age range by gender')
axes[0].tick_params(axis='x', rotation=45)

cohort['country'].value_counts().head(10).sort_values().plot.barh(ax=axes[1])
axes[1].set_title('Top 10 countries')
axes[1].set_xlabel('participants')

sns.histplot(cohort['bmi'].dropna(), bins=25, ax=axes[2])
axes[2].set_title('BMI (derived from height and weight)')
plt.tight_layout()
plt.show()

## 8. Table Grain and Primary Keys

Each table records its data at a different **grain**, meaning what a single row represents. Some tables have one row per person, some one row per day, and some one row per sensor sample. You need to know the grain before joining tables, because joining at mismatched grains silently duplicates or drops rows.

For each table we write down the expected grain and a candidate key (the columns that should uniquely identify a row), then check whether the key really is unique. Duplicate keys can mean repeated uploads or device syncs, conflicting readings for the same moment, or a grain finer than we assumed. For each table with duplicates, we print examples so we can tell which case applies. For example, two heart rate values at the same second, or the same sleep period shifted by a time-zone offset.

In [ ]:
# What one row represents in each table, and whether that key is actually unique
table_grain = {
    'Participants':       (df_participants, ['user_code'],                         'one person'),
    'Blood Pressure':     (df_bp,           ['user_code', 'measurement_datetime'], 'one BP reading'),
    'Heart Rate':         (df_hr,           ['user_code', 'datetime'],             'one heart rate sample'),
    'HRV Measurements':   (df_hrv,          ['rr_code'],                           'one HRV recording session'),
    'Sleep':              (df_sleep,        ['user_code', 'day'],                  'one sleep period'),
    'Wearables':          (df_wearables,    ['user_code', 'day'],                  'one person-day device summary'),
    'Weather':            (df_weather,      ['user_code', 'day'],                  "one person-day of local weather"),
    'Surveys':            (df_surveys,      ['user_code', 'scale', 'created_at'],  'one answer to one question'),
    'Scale Descriptions': (df_scales,       ['Scale', 'Value'],                    'one answer option (codebook)'),
}

grain_rows = []
for name, (df, key, grain) in table_grain.items():
    dup_keys = df.duplicated(subset=key).sum()
    grain_rows.append({
        'Table': name,
        'Row represents': grain,
        'Candidate key': ' + '.join(key),
        'Rows': len(df),
        'Participants': df['user_code'].nunique() if 'user_code' in df.columns else np.nan,
        'Duplicate keys': dup_keys,
        'Exact duplicate rows': df.duplicated().sum(),
        'Key is unique': dup_keys == 0,
    })

display(pd.DataFrame(grain_rows))

# Are duplicated keys exact copies, or conflicting values for the same moment?
for name in ['Heart Rate', 'Sleep', 'Surveys']:
    df, key, _ = table_grain[name]
    dups = df[df.duplicated(subset=key, keep=False)].sort_values(key)
    print(f'\n{name}: {len(dups)} rows share a key')
    display(dups.head(6))

## 9. Linking Tables to the Cohort

This dataset is **relational**. `participants` is the parent table, and every measurement table is a child table linked to it by `user_code`. Two structural questions follow:

1. **Referential integrity:** does every `user_code` in a child table exist in `participants`? A code with no matching participant is an orphan and cannot be tied to demographics.
2. **Coverage:** which data sources does each participant actually have? Not everyone owns a BP cuff, a smartwatch, or a sleep tracker, so the usable sample shrinks as more tables are combined.

The coverage matrix below has one row per participant and one column per table. Each cell shows whether that person has any records in that table.

In [ ]:
cohort_ids = set(df_participants['user_code'])
linked_tables = {
    name: df for name, df in datasets.items()
    if 'user_code' in df.columns and name != 'Participants'
}

integrity = []
for name, df in linked_tables.items():
    ids = set(df['user_code'])
    integrity.append({
        'Table': name,
        'Participants with data': len(ids),
        '% of cohort': round(len(ids) / len(cohort_ids) * 100, 1),
        'Orphan user_codes': len(ids - cohort_ids),
    })
display(pd.DataFrame(integrity).sort_values('Participants with data', ascending=False))

# Coverage matrix: participants x tables, number of records
coverage = (
    pd.DataFrame({name: df.groupby('user_code').size() for name, df in linked_tables.items()})
    .reindex(sorted(cohort_ids))
    .fillna(0)
    .astype(int)
)
has_data = coverage > 0

print('Number of data sources per participant:')
display(has_data.sum(axis=1).value_counts().sort_index().rename_axis('sources').rename('participants').to_frame().T)

combos = has_data.apply(lambda row: ' + '.join(row.index[row]) or '(no linked data)', axis=1)
print('Most common combinations of data sources:')
display(combos.value_counts().head(8).rename('participants').to_frame())

# Sort the columns by coverage and the rows by pattern so that blocks of similar participants group together
col_order = has_data.sum().sort_values(ascending=False).index
plot_matrix = has_data[col_order].sort_values(list(col_order), ascending=False).astype(int)

plt.figure(figsize=(8, 9))
sns.heatmap(plot_matrix, cbar=False, cmap='Blues', yticklabels=False, linewidths=0)
plt.title('Which participants appear in which tables')
plt.ylabel(f'Participants (n = {len(plot_matrix)})')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Longitudinal Structure: Time Span and Sampling Density

Every measurement table is a **time series nested within participants**. The timing differs a lot across tables:

- Heart rate arrives as many samples per day.
- Wearables, sleep, and weather are daily summaries.
- BP, HRV, and surveys are spot measurements, taken whenever the participant chose to record.

The time columns also use different names and formats (`datetime`, `measurement_datetime`, `day`, `created_at`, and some BP timestamps contain a double space). We parse them all into one long table of events, one row per record, and then summarize:

- the calendar window each table covers,
- follow-up length per participant (first to last observation),
- how many records each participant has, which is usually very uneven, and
- when measurements fall relative to each participant's reported symptom onset.

Repeated, unevenly spaced measurements mean rows are **not independent observations**. Summaries and models should account for participant-level clustering.

In [ ]:
time_columns = {
    'Blood Pressure':   (df_bp,        'measurement_datetime'),
    'Heart Rate':       (df_hr,        'datetime'),
    'HRV Measurements': (df_hrv,       'measurement_datetime'),
    'Sleep':            (df_sleep,     'day'),
    'Wearables':        (df_wearables, 'day'),
    'Weather':          (df_weather,   'day'),
    'Surveys':          (df_surveys,   'created_at'),
}

# One long "event" table: table, participant, timestamp
events = pd.concat([
    pd.DataFrame({
        'table': name,
        'user_code': df['user_code'],
        'timestamp': pd.to_datetime(df[col].astype(str).str.replace(r'\s+', ' ', regex=True)),
    })
    for name, (df, col) in time_columns.items()
], ignore_index=True)
events['date'] = events['timestamp'].dt.normalize()

per_user = events.groupby(['table', 'user_code']).agg(
    first=('date', 'min'),
    last=('date', 'max'),
    records=('date', 'size'),
    days_observed=('date', 'nunique'),
)
per_user['follow_up_days'] = (per_user['last'] - per_user['first']).dt.days + 1
per_user['records_per_observed_day'] = per_user['records'] / per_user['days_observed']
per_user['fraction_of_days_observed'] = per_user['days_observed'] / per_user['follow_up_days']

time_summary = (
    events.groupby('table').agg(start=('date', 'min'), end=('date', 'max'), participants=('user_code', 'nunique'))
    .join(per_user.groupby('table')[['records', 'follow_up_days', 'records_per_observed_day',
                                     'fraction_of_days_observed']].median().add_prefix('median_').round(2))
)
display(time_summary)

# Measurements relative to reported symptom onset
onset = cohort.loc[~implausible_onset].set_index('user_code')['symptoms_onset']
events['days_from_onset'] = (events['date'] - events['user_code'].map(onset)).dt.days
onset_summary = events.dropna(subset=['days_from_onset']).groupby('table')['days_from_onset'].agg(
    participants_with_onset=lambda s: events.loc[s.index, 'user_code'].nunique(),
    pct_records_before_onset=lambda s: round((s < 0).mean() * 100, 1),
    median_days_from_onset='median',
)
print('Timing of records relative to symptom onset (participants with a plausible onset date):')
display(onset_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
weekly_active = events.groupby([pd.Grouper(key='date', freq='W'), 'table'])['user_code'].nunique().unstack()
weekly_active.plot(ax=axes[0])
axes[0].set_title('Active participants per week, by table')
axes[0].set_ylabel('participants')

sns.boxplot(data=per_user.reset_index(), x='records', y='table', ax=axes[1])
axes[1].set_xscale('log')
axes[1].set_title('Records per participant (log scale)')
plt.tight_layout()
plt.show()

## 11. Structural Missingness: Which Fields Are Recorded Together

Section 1 counted missing values column by column. In device data, however, missingness usually comes from **structure, not chance**. Different wearables and apps report different fields, so a row tends to be either fully populated for a group of fields or empty for all of them. For example:

- The derived BP indices (Kerdo, Robinson, and so on) exist only when the app calculated them.
- SpO₂ and body temperature appear only for devices that have those sensors.
- Sleep stages exist only for trackers that can detect them.

Below, each row is turned into a presence pattern (which fields are non-null), and we count the most common patterns. We also check how many participants always produce a single pattern. A small number of dominant patterns means the missingness is tied to the device or the app, not to chance. In that case, simple mean imputation would be inappropriate.

In [ ]:
id_time_cols = {'user_code', 'day', 'datetime', 'measurement_datetime', 'sleep_begin', 'sleep_end'}

def missingness_patterns(df, name, top_n=6):
    measure_cols = [c for c in df.columns if c not in id_time_cols]
    present = df[measure_cols].notna()

    pattern_id = present.astype(int).astype(str).apply(''.join, axis=1)
    patterns_per_user = pattern_id.groupby(df['user_code']).nunique()

    print(f'\n{name}: {pattern_id.nunique()} distinct presence patterns across {len(df)} rows')
    print(f'  Top {top_n} patterns cover {pattern_id.value_counts().head(top_n).sum() / len(df):.0%} of rows')
    print(f'  Participants who always produce a single pattern: {(patterns_per_user == 1).mean():.0%}')

    top = present.value_counts().head(top_n)
    pattern_matrix = top.index.to_frame(index=False).astype(int)
    pattern_matrix.index = [f'{n} rows' for n in top.values]

    plt.figure(figsize=(max(6, 0.6 * len(measure_cols)), 0.5 * top_n + 1.5))
    sns.heatmap(pattern_matrix, cbar=False, cmap='Greens', linewidths=0.5, linecolor='white')
    plt.title(f'{name}: most common field presence patterns (green = recorded)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

for name, df in [('Blood Pressure', df_bp), ('Wearables', df_wearables), ('Sleep', df_sleep)]:
    missingness_patterns(df, name)

## 12. Survey Data: Long Format and the Codebook

`surveys.csv` is stored in **long format**, with one row per (participant, question, date) and the answer held in `value`. The questions come from several instruments, identified by the `scale` prefix. For example:

- `S_COVID_*`: COVID symptom check-ins, repeated over time
- `S_HRA_*`: a health risk assessment of chronic conditions, answered roughly once

`scales_description.csv` is the **codebook** that maps each (`Scale`, `Value`) pair to a question description and an answer label. Below we:

- confirm every recorded answer exists in the codebook,
- check whether the survey `text` column just repeats the codebook label,
- summarize each instrument, and
- pivot the COVID symptom check-ins to **wide format** (one row per participant per check-in date), which is the shape most analyses need.

In [ ]:
print('Long-format survey table:', df_surveys.shape)
display(df_surveys.head())

# Join every answer to its codebook entry
surveys_labeled = df_surveys.merge(
    df_scales, left_on=['scale', 'value'], right_on=['Scale', 'Value'], how='left', indicator=True
)
print('Answers matched to the codebook:')
print(surveys_labeled['_merge'].value_counts().to_string())
print(f"\nSurvey 'text' equals codebook 'Meaning' in {(surveys_labeled['text'] == surveys_labeled['Meaning']).mean():.1%} of rows")

# Summarize by instrument (scale prefix)
surveys_labeled['instrument'] = surveys_labeled['scale'].str.extract(r'^(S_[A-Z]+)', expand=False)
instrument_summary = surveys_labeled.groupby('instrument').agg(
    questions=('scale', 'nunique'),
    answers=('scale', 'size'),
    participants=('user_code', 'nunique'),
    distinct_dates=('created_at', 'nunique'),
)
instrument_summary['answers_per_participant'] = (instrument_summary['answers'] / instrument_summary['participants']).round(1)
display(instrument_summary.sort_values('answers', ascending=False))

# Pivot the repeated COVID symptom intensity questions to wide format
covid = surveys_labeled[surveys_labeled['Description'].str.startswith('Symptom intensity', na=False)]
covid_wide = covid.pivot_table(index=['user_code', 'created_at'], columns='scale', values='value', aggfunc='last')
print(f'\nWide COVID symptom check-ins: {covid_wide.shape[0]} participant-dates x {covid_wide.shape[1]} symptoms')
display(covid_wide.head())

response_pct = pd.crosstab(covid['Description'].str.replace('Symptom intensity: ', ''), covid['value'], normalize='index') * 100
plt.figure(figsize=(10, 5))
sns.heatmap(response_pct, annot=True, fmt='.0f', cmap='Reds', cbar_kws={'label': '% of answers'})
plt.title('COVID symptom check-ins: % of answers at each level\n(1 = no symptom, 2 = very mild ... 6 = extremely severe)')
plt.xlabel('value')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 13. Nested Fields in HRV: Raw RR Intervals and Tags

The HRV table packs several levels of data into one row:

- **Summary metrics** (`bpm`, `meanrr`, `sdnn`, `rmssd`, `lf`, `hf`, ...) computed for the whole recording
- **Raw signal** in `rr_data`: the beat-to-beat (RR) intervals in milliseconds, stored as one comma-separated string
- **Context tags** in `tags`: a semicolon-separated list of free-form labels (Workout, Coffee, Illness, ...)
- **Self-reports** (`how_feel`, `how_mood`, `how_sleep`) on a −2 to +2 scale

These strings need to be parsed into arrays or long tables before they can be used. Once `rr_data` is parsed, we can check that the summary columns really are derived from it. Here we compare the reported `meanrr`, `bpm`, `sdnn`, and `rmssd` against values recomputed from the raw intervals. Recording lengths are also examined, since short recordings produce noisier HRV estimates.

In [ ]:
# Parse the nested RR interval strings into numeric arrays
rr_intervals = df_hrv['rr_data'].str.split(',').apply(lambda x: np.array(x, dtype=float))

hrv_check = pd.DataFrame({
    'n_intervals': rr_intervals.apply(len),
    'recording_seconds': rr_intervals.apply(np.sum) / 1000,
    'meanrr_reported': df_hrv['meanrr'],
    'meanrr_from_rr': rr_intervals.apply(np.mean),
    'bpm_reported': df_hrv['bpm'],
    'bpm_from_rr': 60000 / rr_intervals.apply(np.mean),
    'sdnn_reported': df_hrv['sdnn'],
    'sdnn_from_rr': rr_intervals.apply(lambda rr: np.std(rr, ddof=1)),
    'rmssd_reported': df_hrv['rmssd'],
    'rmssd_from_rr': rr_intervals.apply(lambda rr: np.sqrt(np.mean(np.diff(rr) ** 2))),
})

print('RR intervals per recording:')
display(hrv_check['n_intervals'].value_counts().rename_axis('intervals').rename('recordings').to_frame().T)

agreement = pd.DataFrame({
    metric: {
        'correlation': hrv_check[f'{metric}_reported'].corr(hrv_check[f'{metric}_from_rr']),
        'median abs difference': (hrv_check[f'{metric}_reported'] - hrv_check[f'{metric}_from_rr']).abs().median(),
    }
    for metric in ['meanrr', 'bpm', 'sdnn', 'rmssd']
}).T.round(3)
print('Reported summary metrics vs. recomputed from raw rr_data:')
display(agreement)

# Explode the semicolon-separated tags into a long (recording, tag) table
tag_long = (
    df_hrv[['rr_code', 'user_code', 'tags']]
    .assign(tag=df_hrv['tags'].str.split(r';\s*'))
    .explode('tag')
    .dropna(subset=['tag'])
)
print(f"Recordings with at least one tag: {df_hrv['tags'].notna().mean():.1%} | distinct tags: {tag_long['tag'].nunique()}")

print('Self-reported state recorded alongside each HRV session (-2 to +2):')
display(df_hrv[['how_feel', 'how_mood', 'how_sleep']].apply(lambda s: s.value_counts(dropna=False)).sort_index().fillna(0).astype(int))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
example = rr_intervals.iloc[0]
axes[0].plot(np.cumsum(example) / 1000, example, marker='.', linewidth=1)
axes[0].set_title(f"Example raw RR series (rr_code {df_hrv['rr_code'].iloc[0]})")
axes[0].set_xlabel('time (s)')
axes[0].set_ylabel('RR interval (ms)')

sns.histplot(hrv_check['recording_seconds'], bins=40, ax=axes[1])
axes[1].set_title('HRV recording length')
axes[1].set_xlabel('seconds')

tag_long['tag'].value_counts().head(15).sort_values().plot.barh(ax=axes[2])
axes[2].set_title('Most common HRV context tags')
axes[2].set_xlabel('recordings')
plt.tight_layout()
plt.show()